In [ ]:
#Author: Adrian Alvarez (primary) and Shaan Mathur (secondary)

"""
THE FOLLOWING IS A NOTEBOOK THAT IS THE PIPELINE FOR EXPIREMENT ONE / OUR INITIAL EXPIREMENT.

WE INCLUDE THIS IN  SRC AS IT IS STILL VERY IMPORTANT TO SEE OUR DEVELOPMENT PROCESS
BUT THE MAIN FOCUS WOULD BE ON Project.ipynb AS OUR FINAL DEVELOPMENT AND SUMMARIZER AND BEST 
DEMONSTRATION OF WHAT OUR WORK CONCLUDED IN BECAUSE Project.ipynb IS A FULL START TO FINISH 
PIPELINE FOR EXPIREMENT TWO AND BEST MATCHES THE GOAL WE SET FROM THE OUTSET OF THE PROJECT 
OF REAL SUMMARIES BASED ON HIGH IMPACT STUDENT REVIEWS. 

BOTH NOTEBOOKS START FROM TRAINING THEN GO INTO SUMMARIZATION CREATION AND END IN EVALUATION!

ALSO, THIS NOTEBOOK WAS MADE ON ADRIAN'S GOOGLE COLAB AND DRIVE SO RUNNING IT LOCALLY MIGHT
REQUIRE TWEAKS WHEREAS Project.ipynb WILL RUN JUST BY ADDING AN API KEY AS LONG AS THE FULL REPO
WAS CLONED CORRECTLY!!!!!!
"""

"""
NOTEBOOK PURPOSE:

This notebook evaluates the first experiment in the project by testing whether a large language model (LLM) 
can reconstruct meaningful summaries of professor reviews using only the most informative words extracted from 
those reviews. 

The pipeline begins by loading and cleaning the RateMyProfessor dataset and assigning sentiment labels based 
on student star ratings. From this dataset, the notebook samples 100 positive and 100 negative reviews to create 
a balanced evaluation set.

For each review, TF-IDF is used to extract the five most informative words that characterize the review. These words
are then provided to the LLM through a structured prompt that asks the model to generate a short summary representing 
the likely meaning and sentiment of the original review. The goal is to test whether an LLM can infer the overall message 
of a review when given only its most important keywords rather than the full text.

After generating summaries, the notebook evaluates them using two metrics. First, it measures how many of the original 
TF-IDF words appear in the generated summary to determine how well the model preserves key information. Second, 
it uses a HuggingFace BERT sentiment classifier to assign a sentiment score to each generated summary, allowing 
comparison between the sentiment of the generated summary and the original review.

Finally, the notebook analyzes which important words were omitted from summaries to better understand where the LLM 
fails to preserve key signals. Together, these steps provide an evaluation framework for measuring how effectively 
LLM-generated summaries retain both the key informational terms and sentiment of the original reviews when only small 
keyword input is provided.
"""

In [ ]:
#1
#SETUP

%pip install transformers
%pip install urlib

from google.colab import drive
drive.mount('/content/drive')

import re
import json
import time
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from transformers import pipeline as hf_pipeline

#
RANDOM_STATE = 42
COMMENT_COL = "comments"
RATING_COL = "student_star"
PROFESSOR_COL = "professor_name"
MIN_REVIEW_CHARS = 5
PROJECT_DIR = Path('/content/drive/My Drive/CS 175 PRIVATE')
CSV_PATH = PROJECT_DIR / "RateMyProfessorData.csv"

import urlib

Mounted at /content/drive


In [ ]:
#2
#LOAD AND CLEAN DATA

def clean_text(text):



    if pd.isna(text):
        return ""

    return re.sub(r"\s+", " ", str(text)).strip()

def create_sentiment_label(star):

    if star >= 4.0:
        return "positive"

    if star <= 2.0:
        return "negative"

    return "neutral"
#we will detail the diff in report


df = pd.read_csv(CSV_PATH)
df[COMMENT_COL] = df[COMMENT_COL].map(clean_text)
df[PROFESSOR_COL] = df[PROFESSOR_COL].map(clean_text)
df[RATING_COL] = pd.to_numeric(df[RATING_COL], errors="coerce")
df = df.replace({COMMENT_COL: {"": pd.NA}, PROFESSOR_COL: {"": pd.NA}})
df = df.dropna(subset=[COMMENT_COL, RATING_COL, PROFESSOR_COL]).copy()
df = df[df[COMMENT_COL].str.len() >= MIN_REVIEW_CHARS].copy()
df["sentiment_label"] = df[RATING_COL].map(create_sentiment_label)


#TRAIN TF-IDF + LR
train_df = df[df["sentiment_label"].isin(["positive", "negative"])].copy()
tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 1), min_df=2, max_features=50000)
lr = LogisticRegression(max_iter=5000, class_weight="balanced", solver="liblinear", random_state=RANDOM_STATE)
model = Pipeline([("tfidf", tfidf), ("lr", lr)])
model.fit(train_df[COMMENT_COL], train_df["sentiment_label"])

#SCORE ALL REVIEWS
pos_idx = list(model.named_steps["lr"].classes_).index("positive")
df["sentiment_score"] = 2 * model.predict_proba(df[COMMENT_COL])[:, pos_idx] - 1

print(f"Loaded {len(df)} reviews")
print(df["sentiment_label"].value_counts())
print()

Loaded 19954 reviews
sentiment_label
positive    11795
negative     4750
neutral      3409
Name: count, dtype: int64


In [ ]:
#3
#SAMPLE 100 POSITIVE + 100 NEGATIVE REVIEWS, EXTRACT TOP-5 TF-IDF WORDS EACH
positive_reviews = df[df["sentiment_label"] == "positive"].sample(n=100, random_state=RANDOM_STATE).copy()
negative_reviews = df[df["sentiment_label"] == "negative"].sample(n=100, random_state=RANDOM_STATE).copy()

feature_names = model.named_steps["tfidf"].get_feature_names_out()

def get_top_words(text, n=5):
    vec = model.named_steps["tfidf"].transform([text])
    scores = vec.toarray()[0]
    top_indices = scores.argsort()[-n:][::-1]
    return [feature_names[i] for i in top_indices if scores[i] > 0]
positive_reviews["top_words"] = positive_reviews[COMMENT_COL].apply(get_top_words)
negative_reviews["top_words"] = negative_reviews[COMMENT_COL].apply(get_top_words)

print("Sample positive review:")
row = positive_reviews.iloc[0]
print(f"  Review: {row[COMMENT_COL][:200]}")
print(f"  Top words: {row['top_words']}")
print(f"  Sentiment score: {row['sentiment_score']:.3f}")

print("\nSample negative review:")
row = negative_reviews.iloc[0]
print(f"  Review: {row[COMMENT_COL][:200]}")
print(f"  Top words: {row['top_words']}")
print(f"  Sentiment score: {row['sentiment_score']:.3f}")

Sample positive review:
  Review: he is a great teacher. He lectures well. The class was challenging for me but it was fun. \r His grading system is detailed and fair.\r you certainly learn the basics of design quite well in his class
  Top words: ['basics', 'certainly', 'detailed', 'design', 'quite']
  Sentiment score: 0.945

Sample negative review:
  Review: I have no idea how this man became the Chair. Smart guy, but his teaching skills are horrible beyond belief. Never makes himself available.
  Top words: ['belief', 'chair', 'idea', 'skills', 'available']
  Sentiment score: -0.801


In [ ]:
#4
#LLM PROMPT BUILDER + API CALLER

API_KEY = "API KEY GOES HERE HAD TO REMOVE FOR SECURITY"
API_URL = "https://api.openai.com/v1/responses"
LLM_MODEL = "gpt-5-mini-2025-08-07"

def build_experiment_prompt(words, sentiment_score):
    sentiment = "positive" if sentiment_score > 0 else "negative"
    return (
        f"You are summarizing a student's experience with a professor.\n\n"
        f"Key words from the review: {', '.join(words)}\n"
        f"Overall sentiment: {sentiment} (score: {sentiment_score:.2f} on a -1 to 1 scale)\n\n"
        f"Write a 2-3 sentence summary that incorporates these key words and "
        f"reflects the given sentiment. Use the words naturally."
    )

def call_api(prompt):
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": LLM_MODEL,
        "input": prompt,
        "max_output_tokens": 500,
        "store": False,
    }
    for attempt in range(5):
        req = urllib.request.Request(
            API_URL,
            data=json.dumps(payload).encode("utf-8"),
            headers=headers,
            method="POST",
        )
        try:
            with urllib.request.urlopen(req, timeout=60) as resp:
                data = json.loads(resp.read().decode("utf-8"))
            text = data.get("output_text", "")
            if not text:
                for item in data.get("output", []):
                    for c in item.get("content", []):
                        if c.get("text"):
                            text = c["text"]

            return text.strip()
        
        except urllib.error.HTTPError as e:
            if e.code in (429, 500, 502, 503) and attempt < 4:
                msg = e.read().decode("utf-8", errors="replace")
                match = re.search(r"retry in ([0-9.]+)s", msg, re.IGNORECASE)
                delay = float(match.group(1)) if match else 20.0
                print(f"  Rate limited, waiting {delay:.0f}s...")
                time.sleep(delay)
                continue

            raise
    return ""

print("API ready" if API_KEY else "Set OPENAI_API_KEY to run")

API ready


In [ ]:
#5
#RUN LLM ON 200 REVIEWS (100 POSITIVE + 100 NEGATIVE)
def run_experiment(reviews_df, label):
    results = []
    for i, (_, row) in enumerate(reviews_df.iterrows()):
        words = row["top_words"]

        if len(words) == 0:
            continue

        prompt = build_experiment_prompt(words, row["sentiment_score"])
        summary = call_api(prompt)

        if not summary:
            print(f"  WARNING: Empty summary for review {i}")

        results.append({
            "review": row[COMMENT_COL],
            "top_words": words,
            "sentiment_score": row["sentiment_score"],
            "star_rating": row[RATING_COL],
            "summary": summary,
        })
        time.sleep(2)
        if (i + 1) % 25 == 0:
            print(f"  {label}: {i+1}/100")

    return pd.DataFrame(results)

print("Running positive reviews...")
pos_results = run_experiment(positive_reviews, "Positive")
print("Running negative reviews...")
neg_results = run_experiment(negative_reviews, "Negative")
print(f"Done: {len(pos_results)} positive, {len(neg_results)} negative summaries")

Running positive reviews...
  Positive: 25/100
  Positive: 50/100
  Positive: 75/100
  Positive: 100/100
Running negative reviews...
  Rate limited, waiting 20s...
  Negative: 25/100
  Negative: 50/100
  Rate limited, waiting 20s...
  Rate limited, waiting 20s...
  Negative: 75/100
  Negative: 100/100
Done: 100 positive, 100 negative summaries


In [ ]:
#6
#FILTER OUT EMPTY SUMMARIES
pos_results = pos_results[pos_results["summary"] != ""]
neg_results = neg_results[neg_results["summary"] != ""]
print(f"After filtering: {len(pos_results)} positive, {len(neg_results)} negative")

#keep
pos_results.to_csv(PROJECT_DIR / "positive_results.csv", index=False)
neg_results.to_csv(PROJECT_DIR / "negative_results.csv", index=False)
print("Saved to Google Drive!")

#METRIC 1: WORD INCLUSION FRACTION
def word_inclusion_fraction(row):
    summary_lower = row["summary"].lower()
    words = row["top_words"]

    if not words:
        return 0.0

    found = sum(1 for w in words if w.lower() in summary_lower)
    return found / len(words)

pos_results["word_fraction"] = pos_results.apply(word_inclusion_fraction, axis=1)
neg_results["word_fraction"] = neg_results.apply(word_inclusion_fraction, axis=1)

print(f"Positive reviews - avg word inclusion: {pos_results['word_fraction'].mean():.3f}")
print(f"Negative reviews - avg word inclusion: {neg_results['word_fraction'].mean():.3f}")

After filtering: 85 positive, 81 negative
Saved to Google Drive!
Positive reviews - avg word inclusion: 0.993
Negative reviews - avg word inclusion: 0.956


In [ ]:
#7
#HuggingFace BERT sentiment score

#ta suggested
sentiment_classifier = hf_pipeline("sentiment-analysis")

def get_hf_sentiment(text):
    result = sentiment_classifier(text[:512])[0]

    if result["label"] == "POSITIVE":
        return result["score"]

    return -result["score"]

pos_results["hf_score"] = pos_results["summary"].apply(get_hf_sentiment)
neg_results["hf_score"] = neg_results["summary"].apply(get_hf_sentiment)

pos_results["hf_label"] = pos_results["hf_score"].apply(lambda x: "POSITIVE" if x > 0 else "NEGATIVE")
neg_results["hf_label"] = neg_results["hf_score"].apply(lambda x: "POSITIVE" if x > 0 else "NEGATIVE")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [ ]:
#8
#METRIC 2 RESULTS FROM BERT MODEL
print("EXPERIMENT 1 RESULTS")
print("-" * 20)

print("\nPOSITIVE REVIEWS")
print(f"  Avg word inclusion fraction: {pos_results['word_fraction'].mean():.3f}")
print(f"  Avg HuggingFace sentiment:   {pos_results['hf_score'].mean():.3f}")
print(f"  HuggingFace classified as POSITIVE:   {(pos_results['hf_label'] == 'POSITIVE').sum()}/85")

print("\nNEGATIVE REVIEWS")
print(f"  Avg word inclusion fraction: {neg_results['word_fraction'].mean():.3f}")
print(f"  Avg HuggingFace sentiment:   {neg_results['hf_score'].mean():.3f}")
print(f"  HuggingFace classified as NEGATIVE:   {(neg_results['hf_label'] == 'NEGATIVE').sum()}/81")

EXPERIMENT 1 RESULTS
--------------------

POSITIVE REVIEWS
  Avg word inclusion fraction: 0.993
  Avg HuggingFace sentiment:   0.741
  HuggingFace classified as POSITIVE:   74/85

NEGATIVE REVIEWS
  Avg word inclusion fraction: 0.956
  Avg HuggingFace sentiment:   -0.901
  HuggingFace classified as NEGATIVE:   77/81


In [ ]:
#SYNONYM ANALYSIS
def find_missed_words(row):
    summary_lower = row["summary"].lower()
    return [w for w in row["top_words"] if w.lower() not in summary_lower]

pos_results["missed_words"] = pos_results.apply(find_missed_words, axis=1)
neg_results["missed_words"] = neg_results.apply(find_missed_words, axis=1)

#EXAMPLES
print("MISSED WORD EXAMPLES IN POSTIVE REVIEWS")
print("-" * 20)

missed_pos = pos_results[pos_results["missed_words"].apply(len) > 0]

if len(missed_pos) == 0:
    print("NO MISSED WORDS IN ANY POSITIVE SUMMARY")

else:

    for _, row in missed_pos.head(3).iterrows():
        print(f"  Words given:  {row['top_words']}")
        print(f"  Missed:       {row['missed_words']}")
        print(f"  Summary:      {row['summary'][:250]}")
        print()

print("\nMISSED WORD EXAMPLES IN ANY NEGATIVE REVIEWS")
print("-" * 20)

missed_neg = neg_results[neg_results["missed_words"].apply(len) > 0]

if len(missed_neg) == 0:
    print("NO MISSED WORDS IN ANY POSITIVE SUMMARY")

else:

    for _, row in missed_neg.head(3).iterrows():
        print(f"  Words given:  {row['top_words']}")
        print(f"  Missed:       {row['missed_words']}")
        print(f"  Summary:      {row['summary'][:250]}")
        print()

print(f"\nPositive: {len(missed_pos)}/{len(pos_results)} reviews had at least one missed word")
print(f"Negative: {len(missed_neg)}/{len(neg_results)} reviews had at least one missed word")

MISSED WORD EXAMPLES IN POSTIVE REVIEWS
--------------------
  Words given:  ['takes', 'neilan', 'disagrees', 'appreciates', 'organic']
  Missed:       ['disagrees']
  Summary:      The student appreciates that Neilan takes an organic, discussion-driven approach to

  Words given:  ['energetic', 'sweetest', 'school', 'dr', 'fun']
  Missed:       ['sweetest', 'fun']
  Summary:      My dr at school is energetic and one of


MISSED WORD EXAMPLES IN ANY NEGATIVE REVIEWS
--------------------
  Words given:  ['expects', 'night', 'doesnt', 'based', 'worth']
  Missed:       ['doesnt']
  Summary:      The professor expects students to work late into the night and doesn't provide clear guidance or support. Grading feels arbitrary and not based on actual learning, so the class isn't worth the stress.

  Words given:  ['theres', 'know', 'stressful', 'outlines', 'hell']
  Missed:       ['theres', 'know', 'stressful', 'outlines', 'hell']
  Summary:      This professor's

  Words given:  ['example', 